## Fine Tunning del Modelo de Embedding

Como bien sabemos, el mundo de Juego de Tronos es extenso, lleno de nombres propios y significacdos ocultos. Puede ser que el modelo de embedding sea incapaz de pillar relaciones semánticas del mundo que se le presenta.

Por ejemplo, si le hicieramos la pregunta de: ¿Qué tipo de familia es la familia Lannyster?, a lo mejor no sabría decir que es avariciosa, la más rica de poniente...

Por eso mismo, vamos a hacer un fine tunning al modelo de embedding que vamos a utilizar. Luego, evaluaremos las respuestas que conseguimos si hacemos el embedding con el modelo normal y con el modelo fine tunneado.

Para poder hacer el fine tunning, necesitamos muchos tripletes. En los tripletes tendremos lo siguiente:

- Una pregunta
- El chunk de el que se puede obtener la respuesta
- Un chunk que no tiene nada que ver

Como necesitamos miles de tripletes de este estilo, vamos a automatizarlo.

1. Cogemos el libro y hacemos chunks. No lo tokenizamos. Los chunks serán de 6000 caractéres, tendrán un solapamiento de 2 parrafos del anterior, y además respetarán la estructura del libro (capitulos...).
2. Utilizamos el modelo de Llama-3.1 8B
3. Le pedimos que, por cada chunck, nos genere tres dobletes (la pregunta y el chunk utilizado)
4. Guardamos los dobletes en .json para luego generar el triplete final y evaluar

Es decir, al modelo de Llama le pasamos el chunk, y le pedimos que genere tres preguntas basandose en ese chunk.



In [ ]:
"""
fine_tunning_embbeding.py
-----------------
1. Extrae texto del epub y lo divide en chunks
2. Por cada chunk, usa un LLM de HuggingFace para generar preguntas sintéticas
3. Guarda los pares (pregunta, chunk) en un JSON listo para fine-tuning

Dependencias:
    pip install ebooklib beautifulsoup4 transformers torch accelerate bitsandbytes

Uso:
    python generate_pairs.py --epub libro.epub --output pairs.json
"""

In [ ]:
# =========================
# INSTALL
# =========================
!pip install ebooklib -q
!pip install bs4 -q
!pip install torch -q
!pip install -q -U bitsandbytes
!pip install -q -U transformers accelerate
!pip install sentence-transformers -q
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 155.7 MB/s eta 0:00:0000:010:01
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.6 MB/s eta 0:00:0000:0100:01


In [ ]:
# =========================
# IMPORTS
# =========================

import re
import json
import random
import os
import gc
import zipfile
import functools
import numpy as np
from tqdm.notebook import tqdm

from bs4 import BeautifulSoup

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

from google.colab import drive
from huggingface_hub import login
import faiss

In [ ]:
# =========================
# CONFIG
# =========================

EPUB_PATH   = "/content/drive/MyDrive/NLP_PRACTICA/1.epub"
OUTPUT_PATH = "/content/drive/MyDrive/BIG DATA/pairs.json"

# Nuevo chunking
TARGET_CHARS  = 6000
OVERLAP_PARAS = 2

QUESTIONS_PER_CHUNK = 3

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

PROMPT_TEMPLATE = """Dado el siguiente fragmento de una novela de fantasía en español, genera exactamente {n} preguntas cuya respuesta esté contenida únicamente en ese fragmento. Las preguntas deben ser específicas, variadas y en español. No incluyas las respuestas.

Fragmento:
{chunk}

Devuelve SOLO las preguntas numeradas, sin ningún texto adicional. Ejemplo de formato:
1. ¿Pregunta uno?
2. ¿Pregunta dos?
3. ¿Pregunta tres?"""

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Mejor no hardcodear el token directamente en el notebook.
# Opción 1: login() y pegar token cuando lo pida.
# Opción 2: login("tu_token")
login()

drive.mount("/content/drive")

login("")

print = functools.partial(print, flush=True)

In [ ]:
# ============================================================================
# 1. EXTRACCIÓN DEL EPUB RESPETANDO ESTRUCTURA
# ============================================================================

def extract_full_text(epub_path: str) -> str:
    """
    Extrae el texto del EPUB leyendo los archivos XHTML en orden.
    Mantiene saltos de línea para poder recuperar mejor párrafos y capítulos.
    """
    z = zipfile.ZipFile(epub_path)

    files = sorted(
        f for f in z.namelist()
        if f.endswith(".xhtml") or f.endswith(".html")
    )

    parts = []

    for f in files:
        html = z.read(f).decode("utf-8", errors="ignore")
        soup = BeautifulSoup(html, "html.parser")

        # El separator="\n" es importante para conservar bloques.
        text = soup.get_text("\n", strip=True)

        if text:
            parts.append(text)

    return "\n".join(parts)


def normalize_line_breaks(text: str) -> str:
    """
    Limpieza suave: no destruye todos los saltos de línea.
    """
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

Mounted at /content/drive


In [ ]:
# ============================================================================
# 2. DETECCIÓN DE CAPÍTULOS
# ============================================================================

def split_into_chapters(text: str) -> list[dict]:
    """
    Detecta capítulos con formato tipo:

    BRAN (1)
    CATELYN (2)
    JON (3)

    Devuelve una lista de diccionarios con título, POV, orden y texto.
    Si no encuentra capítulos, devuelve todo el libro como un único bloque.
    """

    pattern = r"\n([A-ZÁÉÍÓÚÑÜ ]+\s\(\d+\))\n"
    matches = list(re.finditer(pattern, text))

    chapters = []

    if not matches:
        return [{
            "title": "LIBRO COMPLETO",
            "pov": "",
            "text": text.strip(),
            "order": 1
        }]

    for i, match in enumerate(matches):
        title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        body = text[start:end].strip()
        pov = title.split("(")[0].strip()

        if body:
            chapters.append({
                "title": title,
                "pov": pov,
                "text": body,
                "order": i + 1
            })

    return chapters

In [ ]:
# ============================================================================
# 3. PÁRRAFOS Y CHUNKING SEMÁNTICO
# ============================================================================

def split_into_paragraphs(text: str) -> list[str]:
    """
    Intenta recuperar párrafos reales a partir de los saltos del EPUB.
    Si salen pocos bloques, usa un fallback por frases.
    """

    text = normalize_line_breaks(text)

    # Primer intento: usar saltos reales del EPUB.
    raw_paras = [
        re.sub(r"\s+", " ", p).strip()
        for p in text.split("\n")
        if p.strip()
    ]

    # Filtramos bloques demasiado cortos que suelen ser ruido.
    paras = [p for p in raw_paras if len(p) > 20]

    if len(paras) > 3:
        return paras

    # Fallback: dividir por final de frase seguido de mayúscula, raya o comillas.
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑÜ—«"])',
        re.sub(r"\s+", " ", text)
    )

    return [s.strip() for s in sentences if len(s.strip()) > 20]


def split_long_paragraph(para: str, target_chars: int) -> list[str]:
    """
    Divide un párrafo enorme por frases si supera target_chars.
    """
    sentences = re.split(r'(?<=[.!?])\s+', para)
    pieces = []

    current = []
    current_len = 0

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if current and current_len + len(sent) > target_chars:
            pieces.append(" ".join(current))
            current = []
            current_len = 0

        current.append(sent)
        current_len += len(sent) + 1

    if current:
        pieces.append(" ".join(current))

    return pieces


def chunk_chapter_by_paragraphs(
    chapter: dict,
    target_chars: int = TARGET_CHARS,
    overlap_paras: int = OVERLAP_PARAS
) -> list[dict]:
    """
    Crea chunks dentro de un capítulo.
    No mezcla capítulos distintos.
    Solapa los últimos `overlap_paras` párrafos del chunk anterior.
    """

    paragraphs = split_into_paragraphs(chapter["text"])

    # Partimos párrafos demasiado grandes.
    expanded_paragraphs = []
    for para in paragraphs:
        if len(para) > target_chars:
            expanded_paragraphs.extend(split_long_paragraph(para, target_chars))
        else:
            expanded_paragraphs.append(para)

    chunks = []
    i = 0

    while i < len(expanded_paragraphs):
        current_paras = []
        current_len = 0

        while i < len(expanded_paragraphs):
            para = expanded_paragraphs[i]
            extra_len = len(para) + 1

            if current_paras and current_len + extra_len > target_chars:
                break

            current_paras.append(para)
            current_len += extra_len
            i += 1

        if current_paras:
            chunk_text = "\n\n".join(current_paras).strip()

            chunks.append({
                "text": chunk_text,
                "paragraphs": list(current_paras),
                "chapter_title": chapter["title"],
                "chapter_pov": chapter["pov"],
                "chapter_order": chapter["order"]
            })

        # Retrocedemos para meter overlap en el siguiente chunk.
        # Solo si todavía quedan párrafos por procesar.
        if overlap_paras > 0 and i < len(expanded_paragraphs):
            i = max(0, i - overlap_paras)

    return chunks


def build_chunks_from_epub(
    epub_path: str,
    target_chars: int = TARGET_CHARS,
    overlap_paras: int = OVERLAP_PARAS
) -> list[dict]:
    """
    Pipeline completo:
    EPUB -> texto -> capítulos -> chunks con metadatos.
    """

    raw_text = extract_full_text(epub_path)
    text = normalize_line_breaks(raw_text)

    print(f"Texto extraído: {len(text):,} caracteres")

    chapters = split_into_chapters(text)
    print(f"Capítulos detectados: {len(chapters)}")

    all_chunks = []

    for chapter in chapters:
        chapter_chunks = chunk_chapter_by_paragraphs(
            chapter=chapter,
            target_chars=target_chars,
            overlap_paras=overlap_paras
        )

        all_chunks.extend(chapter_chunks)

    # Añadimos chunk_id global.
    for idx, chunk in enumerate(all_chunks):
        chunk["chunk_id"] = idx

    return all_chunks

In [ ]:
# ============================================================================
# 4. CARGA DEL MODELO
# ============================================================================

def load_model(model_name: str):
    print(f"Cargando modelo {model_name} en 4-bit...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model.eval()

    return tokenizer, model

In [ ]:
# ============================================================================
# 5. GENERACIÓN DE PREGUNTAS
# ============================================================================

def parse_questions(response: str) -> list[str]:
    """
    Extrae preguntas numeradas de la respuesta del modelo.
    Acepta:
    1. pregunta
    1) pregunta
    1- pregunta
    """

    lines = response.strip().split("\n")
    questions = []

    for line in lines:
        line = line.strip()

        match = re.match(r"^\d+[\.\)\-]\s*(.+)", line)

        if match:
            q = match.group(1).strip()

            if len(q) > 10:
                questions.append(q)

    return questions


def generate_questions(
    chunk: str,
    tokenizer,
    model,
    n: int = QUESTIONS_PER_CHUNK
) -> list[str]:

    prompt = PROMPT_TEMPLATE.format(n=n, chunk=chunk)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=300,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=False,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    del input_ids, attention_mask, output_ids, new_tokens
    torch.cuda.empty_cache()

    return parse_questions(response)

In [ ]:
# ============================================================================
# 6. EJECUCIÓN PRINCIPAL
# ============================================================================

print("Construyendo chunks desde el EPUB...")

chunks = build_chunks_from_epub(
    epub_path=EPUB_PATH,
    target_chars=TARGET_CHARS,
    overlap_paras=OVERLAP_PARAS
)

print(f"Chunks generados: {len(chunks)}")
print(f"Tamaño objetivo: ~{TARGET_CHARS} caracteres")
print(f"Overlap: {OVERLAP_PARAS} párrafos")

print("\nEjemplo de primer chunk:")
print("Chunk ID:", chunks[0]["chunk_id"])
print("Capítulo:", chunks[0]["chapter_title"])
print("POV:", chunks[0]["chapter_pov"])
print("Texto:", chunks[0]["text"][:500], "...")

In [ ]:
tokenizer, model = load_model(MODEL_NAME)

pairs = []
errors = []

for i, chunk_data in enumerate(chunks):
    chunk_text = chunk_data["text"]

    try:
        questions = generate_questions(
            chunk=chunk_text,
            tokenizer=tokenizer,
            model=model,
            n=QUESTIONS_PER_CHUNK
        )

        for q in questions:
            pairs.append({
                "question": q,
                "chunk": chunk_text,
                "chunk_id": chunk_data["chunk_id"],
                "chapter_title": chunk_data["chapter_title"],
                "chapter_pov": chunk_data["chapter_pov"],
                "chapter_order": chunk_data["chapter_order"]
            })

            print(f"[Chunk {i + 1}/{len(chunks)}] Q: {q}")
            print(f"Capítulo: {chunk_data['chapter_title']}")
            print(f"C: {chunk_text[:180]}...")
            print()

    except Exception as e:
        errors.append({
            "chunk_id": chunk_data["chunk_id"],
            "chapter_title": chunk_data["chapter_title"],
            "error": str(e)
        })

        print(f"[ERROR] Chunk {i}: {e}")
        continue

    # Guardado parcial cada 100 chunks
    if (i + 1) % 100 == 0:
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(pairs, f, ensure_ascii=False, indent=2)

        print(f"[Checkpoint] {len(pairs)} pares guardados tras chunk {i + 1}")


print(f"\nPares generados: {len(pairs)}")
print(f"Errores: {len(errors)}")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)

print(f"Guardado en: {OUTPUT_PATH}")

Una vez hechas las duplas, toca construir los tripletes. La idea es poner chunks que no respondan a las preguntas que se plantean. No obstante, no va a ser tan sencillo como poner un chunk aleatorio cualquiera (excepto el correcto). Se van a introducir dos chunks:

- El random, que supondrá un 30% de los chunks negativos
- El "hard", que supondrá el otro 70%.

El 30% de las preguntas tendrán chunks aleatorios escogidos al azar. Estos chunks no valdrán para poder responder la pregunta que se plantea. Además, al ser aleatorios, puede ser que hagamos una pregunta sobre Tyrion y que el chunk negativo hable sobre Jon Nieve. Esto hara que para el embedding, sea fácil de distinguirlos.

Por otro lado, el otro 70% serán chunks "hard", o semanticamente cercanos. Por ejemplo, si preguntamos por Tyrion, buscaremos chunks que hablen sobre Tyrion pero que no sean capaces de responder a la pregunta.

La siguiente pregunta es: ¿Como encontramos esos chunks cercanos?

Usaremos el modelo de embedding que vamos a fine-tunnear. Buscaremos los chunks más cercanos en distancia coseno al chunk correcto de referencia. Entonces cogeremos el más cecano de todos y lo pondremos como el chunk negativo.

De esta forma, nos aseguramos de que el modelo aprenda bien, pero sin llegar a tener overfitting (para eso introducimos los random).



In [ ]:
# ============================================================================
# 0. LIBERAR MEMORIA DEL LLM ANTES DE CARGAR EMBEDDINGS
# ============================================================================

for var_name in ["model", "tokenizer", "llm_model", "llm_tokenizer"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memoria liberada tras borrar el modelo generador.")

Memoria liberada tras borrar el modelo generador.


In [ ]:
# ============================================================================
# 1. CARGAR PAIRS DESDE JSON
# ============================================================================

PAIRS_PATH = "/content/drive/MyDrive/BIG DATA/pairs.json"
TRIPLETS_PATH = "/content/drive/MyDrive/BIG DATA/triplets.json"

EMBEDDING_NAME = "Qwen/Qwen3-Embedding-8B"

BATCH_SIZE = 32
HARD_POOL = 10
HARD_RATIO = 0.7

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

with open(PAIRS_PATH, "r", encoding="utf-8") as f:
    pairs = json.load(f)

print(f"Pairs cargados: {len(pairs)}")

if len(pairs) == 0:
    raise ValueError("El JSON de pairs está vacío.")

Pairs cargados: 2670


In [ ]:
# ============================================================================
# 2. VALIDAR FORMATO NUEVO DE PAIRS
# ============================================================================

required_keys = {
    "question",
    "chunk",
    "chunk_id",
    "chapter_title",
    "chapter_pov",
    "chapter_order"
}

missing_examples = []

for i, p in enumerate(pairs[:20]):
    missing = required_keys - set(p.keys())
    if missing:
        missing_examples.append((i, missing))

if missing_examples:
    raise ValueError(
        "Algunos pairs no tienen el formato nuevo con metadatos de capítulo. "
        f"Ejemplos: {missing_examples}"
    )

print("Formato validado correctamente.")
print("Ejemplo de pair:")
print(json.dumps(pairs[0], ensure_ascii=False, indent=2)[:1000])

Número de pares disponibles: 2670


In [ ]:
# ============================================================================
# 3. RECONSTRUIR CHUNKS ÚNICOS CON METADATOS
# ============================================================================

chunks_by_id = {}

for p in pairs:
    cid = p["chunk_id"]

    if cid not in chunks_by_id:
        chunks_by_id[cid] = {
            "chunk_id": cid,
            "text": p["chunk"],
            "chapter_title": p.get("chapter_title", ""),
            "chapter_pov": p.get("chapter_pov", ""),
            "chapter_order": p.get("chapter_order", None)
        }

# Orden estable por chunk_id
chunk_ids = sorted(chunks_by_id.keys())
chunk_texts = [chunks_by_id[cid]["text"] for cid in chunk_ids]
id_to_idx = {cid: idx for idx, cid in enumerate(chunk_ids)}

print(f"Chunks únicos para indexar: {len(chunk_texts)}")
print(f"Primer chunk_id: {chunk_ids[0]}")
print(f"Último chunk_id: {chunk_ids[-1]}")

print("\nEjemplo de chunk único:")
example_id = chunk_ids[0]
print(json.dumps(chunks_by_id[example_id], ensure_ascii=False, indent=2)[:1000])

Chunks únicos para indexar: 890


In [ ]:
# ============================================================================
# 4. CARGAR MODELO DE EMBEDDINGS Y EMBEDDEAR CHUNKS
# ============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Cargando modelo de embeddings {EMBEDDING_NAME} en {device}...")

emb_model = SentenceTransformer(
    EMBEDDING_NAME,
    device=device
)

print(f"Embeddeando {len(chunk_texts)} chunks...")

embeddings = emb_model.encode(
    chunk_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)

print(f"Embeddings shape: {embeddings.shape}")

Cargando modelo de embeddings BAAI/bge-m3 en cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embeddeando 890 chunks...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embeddings shape: (890, 1024)


In [ ]:
# ============================================================================
# 5. CONSTRUIR ÍNDICE FAISS
# ============================================================================

dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"Índice FAISS construido con {index.ntotal} vectores")

Índice FAISS construido con 890 vectores


In [ ]:
# ============================================================================
# 6. FUNCIONES AUXILIARES PARA NEGATIVOS
# ============================================================================

def are_adjacent_or_same_chunk(chunk_id_a, chunk_id_b, max_distance=1):
    """
    Evita usar como negativo el propio chunk o chunks inmediatamente contiguos.
    Como ahora hay overlap de párrafos, los chunks vecinos pueden compartir texto.
    """
    return abs(int(chunk_id_a) - int(chunk_id_b)) <= max_distance


def same_chapter(chunk_id_a, chunk_id_b):
    """
    Comprueba si dos chunks pertenecen al mismo capítulo.
    """
    return (
        chunks_by_id[chunk_id_a]["chapter_order"]
        == chunks_by_id[chunk_id_b]["chapter_order"]
    )


def is_valid_negative(positive_chunk_id, negative_chunk_id):
    """
    Criterios para aceptar un negativo:
    - No puede ser el mismo chunk.
    - No puede ser un chunk inmediatamente anterior o posterior.
    - Si es del mismo capítulo y muy cercano, se descarta.

    Esto es importante porque ahora los chunks tienen overlap real.
    """

    if positive_chunk_id == negative_chunk_id:
        return False

    if are_adjacent_or_same_chunk(positive_chunk_id, negative_chunk_id, max_distance=1):
        return False

    return True


def get_random_negative(chunk_id, max_attempts=100):
    """
    Devuelve un chunk_id aleatorio válido como negativo.
    """

    for _ in range(max_attempts):
        neg_id = random.choice(chunk_ids)

        if is_valid_negative(chunk_id, neg_id):
            return neg_id

    # Fallback más permisivo si el corpus es muy pequeño
    candidates = [cid for cid in chunk_ids if cid != chunk_id]

    if not candidates:
        raise ValueError("No hay candidatos negativos disponibles.")

    return random.choice(candidates)


def get_hard_negative(chunk_id):
    """
    Devuelve un negativo semánticamente cercano pero distinto.
    Usa FAISS para buscar vecinos cercanos.

    Como ahora los chunks tienen overlap, descartamos:
    - el mismo chunk;
    - chunks contiguos.
    """

    idx = id_to_idx[chunk_id]
    query_vec = embeddings[idx].reshape(1, -1)

    k = min(HARD_POOL + 5, len(chunk_ids))

    _, indices = index.search(query_vec, k)

    for retrieved_idx in indices[0]:
        candidate_id = chunk_ids[int(retrieved_idx)]

        if is_valid_negative(chunk_id, candidate_id):
            return candidate_id

    return get_random_negative(chunk_id)

In [ ]:
# ============================================================================
# 7. GENERAR TRIPLETS CON METADATOS
# ============================================================================

triplets = []

for pair in tqdm(pairs, desc="Generando triplets"):
    positive_chunk_id = pair["chunk_id"]

    if random.random() < HARD_RATIO:
        negative_chunk_id = get_hard_negative(positive_chunk_id)
        negative_type = "hard"
    else:
        negative_chunk_id = get_random_negative(positive_chunk_id)
        negative_type = "random"

    negative_chunk = chunks_by_id[negative_chunk_id]

    triplets.append({
        "anchor": pair["question"],

        "positive": pair["chunk"],
        "negative": negative_chunk["text"],

        "positive_chunk_id": positive_chunk_id,
        "negative_chunk_id": negative_chunk_id,

        "positive_chapter_title": pair.get("chapter_title", ""),
        "positive_chapter_pov": pair.get("chapter_pov", ""),
        "positive_chapter_order": pair.get("chapter_order", None),

        "negative_chapter_title": negative_chunk.get("chapter_title", ""),
        "negative_chapter_pov": negative_chunk.get("chapter_pov", ""),
        "negative_chapter_order": negative_chunk.get("chapter_order", None),

        "negative_type": negative_type
    })

print(f"Triplets generados: {len(triplets)}")

Generando triplets:   0%|          | 0/2670 [00:00<?, ?it/s]

Triplets generados: 2670


In [ ]:
# ============================================================================
# 8. GUARDAR TRIPLETS
# ============================================================================

os.makedirs(os.path.dirname(TRIPLETS_PATH), exist_ok=True)

with open(TRIPLETS_PATH, "w", encoding="utf-8") as f:
    json.dump(triplets, f, ensure_ascii=False, indent=2)

print(f"Triplets guardados en: {TRIPLETS_PATH}")

Guardado en: /content/drive/MyDrive/BIG DATA/triplets.json

Ejemplos de triplets:

A (pregunta) : ¿Cuál era el significado de la palabra "bastardo" para Sansa desde que tenía edad y uso de razón?
P (correcto) : y curioso, que siempre quería seguirlos y participar en cualquier cosa que hicieran Robb y Jon. También echaba de menos a las chicas, incluso a Sansa, que jamás lo había llamado de otra manera que no ...
N (negativo) : un latigazo, pero Jon no gritó. —Menuda boca tiene el señorito —dijo Sapo acercándose un poco más. Tenía ojillos porcinos, pequeños y brillantes—. ¿La boquita la sacaste de tu mamá, bastardo? ¿De qué ...
--------------------------------------------------------------------------------
A (pregunta) : ¿Por qué Tyrion Lannister había sido instalado en habitaciones de la Torre del Rey?
P (correcto) : mirándolo desde abajo. Sentía como si el peso de todo aquel hielo cayera sobre él, como si estuviera a punto de derrumbarse. Y el muchacho tenía la intuición de que, si el

In [ ]:
# ============================================================================
# 9. PREVIEW DE TRIPLETS
# ============================================================================

print("\nEjemplos de triplets:\n")

for t in random.sample(triplets, min(3, len(triplets))):
    print(f"A pregunta:")
    print(t["anchor"])

    print("\nP correcto:")
    print(f"Chunk ID: {t['positive_chunk_id']}")
    print(f"Capítulo: {t['positive_chapter_title']}")
    print(t["positive"][:300], "...")

    print("\nN negativo:")
    print(f"Chunk ID: {t['negative_chunk_id']}")
    print(f"Capítulo: {t['negative_chapter_title']}")
    print(f"Tipo negativo: {t['negative_type']}")
    print(t["negative"][:300], "...")

    print("-" * 100)

Pairs cargados desde JSON: 2670
Chunks únicos en corpus: 890
Train chunks: 712
Test chunks:  178
Train pairs:  2136
Test pairs:   534


In [ ]:
# ============================================================================
# 10. PREPARACIÓN DE EVALUACIÓN
# ============================================================================

PAIRS_PATH = "/content/drive/MyDrive/BIG DATA/pairs.json"

EVAL_SEED = 42
TEST_SIZE = 0.2
EVAL_BATCH_SIZE = 32
TOP_KS = [1, 3, 5, 10]

with open(PAIRS_PATH, "r", encoding="utf-8") as f:
    pairs_eval = json.load(f)

print(f"Pairs cargados desde JSON: {len(pairs_eval)}")

if len(pairs_eval) == 0:
    raise ValueError("pairs_eval está vacío.")

Entrenando en: cuda
Triplets antes del filtrado:  1681
Triplets después del filtrado: 1661
Eliminados: 20


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Warmup steps: 20
Batch size:   8
Epochs:       1
LR:           2e-05


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo guardado en: /content/drive/MyDrive/BIG DATA/bge-m3-finetuned-got_3


In [ ]:
# ============================================================================
# 11. RECONSTRUIR CORPUS DE CHUNKS ÚNICOS
# ============================================================================

chunks_by_id = {}

for p in pairs_eval:
    cid = p["chunk_id"]

    if cid not in chunks_by_id:
        chunks_by_id[cid] = {
            "chunk_id": cid,
            "text": p["chunk"],
            "chapter_title": p.get("chapter_title", ""),
            "chapter_pov": p.get("chapter_pov", ""),
            "chapter_order": p.get("chapter_order", None)
        }

chunk_ids = sorted(chunks_by_id.keys())
chunk_texts = [chunks_by_id[cid]["text"] for cid in chunk_ids]

print(f"Chunks únicos en corpus: {len(chunk_ids)}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Evaluando: fine_tuned
Embeddeando corpus...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embeddeando preguntas de test...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]


Resultados fine_tuned:
MRR@1: 0.1723
MRR@3: 0.2266
MRR@5: 0.2432
MRR@10: 0.2572
Recall@1: 0.1723
Recall@3: 0.2959
Recall@5: 0.3689
Recall@10: 0.4794

Comparación final:
MRR@1: baseline=0.3277 | fine_tuned=0.1723 | delta=-0.1554
MRR@3: baseline=0.4145 | fine_tuned=0.2266 | delta=-0.1879
MRR@5: baseline=0.4322 | fine_tuned=0.2432 | delta=-0.1890
MRR@10: baseline=0.4455 | fine_tuned=0.2572 | delta=-0.1882
Recall@1: baseline=0.3277 | fine_tuned=0.1723 | delta=-0.1554
Recall@3: baseline=0.5243 | fine_tuned=0.2959 | delta=-0.2285
Recall@5: baseline=0.6011 | fine_tuned=0.3689 | delta=-0.2322
Recall@10: baseline=0.7004 | fine_tuned=0.4794 | delta=-0.2210


In [ ]:
# ============================================================================
# 12. SPLIT POR CAPÍTULO PARA EVITAR LEAKAGE
# ============================================================================

chapter_orders = sorted(
    {
        p["chapter_order"]
        for p in pairs_eval
        if p.get("chapter_order") is not None
    }
)

rng = random.Random(EVAL_SEED)
rng.shuffle(chapter_orders)

n_test = max(1, int(len(chapter_orders) * TEST_SIZE))

test_chapter_orders = set(chapter_orders[:n_test])
train_chapter_orders = set(chapter_orders[n_test:])

train_pairs_eval = [
    p for p in pairs_eval
    if p.get("chapter_order") in train_chapter_orders
]

test_pairs_eval = [
    p for p in pairs_eval
    if p.get("chapter_order") in test_chapter_orders
]

train_chunk_ids = {
    p["chunk_id"]
    for p in train_pairs_eval
}

test_chunk_ids = {
    p["chunk_id"]
    for p in test_pairs_eval
}

print(f"Train capítulos: {len(train_chapter_orders)}")
print(f"Test capítulos:  {len(test_chapter_orders)}")

print(f"Train chunks:    {len(train_chunk_ids)}")
print(f"Test chunks:     {len(test_chunk_ids)}")

print(f"Train pairs:     {len(train_pairs_eval)}")
print(f"Test pairs:      {len(test_pairs_eval)}")

if len(test_pairs_eval) == 0:
    raise ValueError("No hay pairs de test. Baja TEST_SIZE o revisa chapter_order.")

In [ ]:
# ============================================================================
# 13. PREPARAR QUERIES DE TEST
# ============================================================================

test_questions = [p["question"] for p in test_pairs_eval]
test_true_ids = [p["chunk_id"] for p in test_pairs_eval]

print(f"Preguntas de test: {len(test_questions)}")

In [ ]:
# ============================================================================
# 14. FUNCIÓN DE EVALUACIÓN
# ============================================================================

def evaluate_model(model_ref, label, batch_size=EVAL_BATCH_SIZE, top_ks=TOP_KS):
    """
    model_ref puede ser:
    - nombre de Hugging Face;
    - path local de modelo fine-tuneado.

    Evalúa retrieval pregunta -> chunk correcto.
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(
        model_ref,
        device=device
    )

    print(f"\nEvaluando: {label}")
    print(f"Modelo: {model_ref}")
    print(f"Device: {device}")

    print("\nEmbeddeando corpus completo de chunks...")

    corpus_embeddings = model.encode(
        chunk_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    print("\nEmbeddeando preguntas de test...")

    query_embeddings = model.encode(
        test_questions,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    sims = query_embeddings @ corpus_embeddings.T

    max_k = max(top_ks)
    top_idx = np.argsort(-sims, axis=1)[:, :max_k]

    results = {}
    total = len(test_true_ids)

    for k in top_ks:
        hits = 0
        mrr = 0.0

        for i in range(total):
            retrieved_ids = [
                chunk_ids[j]
                for j in top_idx[i, :k]
            ]

            true_id = test_true_ids[i]

            if true_id in retrieved_ids:
                hits += 1
                rank = retrieved_ids.index(true_id) + 1
                mrr += 1.0 / rank

        results[f"Recall@{k}"] = hits / total
        results[f"MRR@{k}"] = mrr / total

    print(f"\nResultados {label}:")

    ordered_metrics = sorted(
        results.keys(),
        key=lambda x: (x.split("@")[0], int(x.split("@")[1]))
    )

    for metric in ordered_metrics:
        print(f"{metric}: {results[metric]:.4f}")

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [ ]:
# ============================================================================
# 15. EVALUAR BASELINE
# ============================================================================

baseline_results = evaluate_model(
    EMBEDDING_NAME,
    "baseline"
)

In [ ]:
# ============================================================================
# 16. PREPARACIÓN DEL FINE-TUNING
# ============================================================================

TRIPLETS_PATH = "/content/drive/MyDrive/BIG DATA/triplets.json"
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/BIG DATA/qwen3-embedding-8b-finetuned-got_bien"

FT_BATCH_SIZE = 1
FT_EPOCHS = 2
FT_LEARNING_RATE = 2e-5
FT_WARMUP_RATIO = 0.1

if "train_chapter_orders" not in globals():
    raise ValueError("Primero ejecuta la preparación de evaluación para crear train_chapter_orders.")

with open(TRIPLETS_PATH, "r", encoding="utf-8") as f:
    all_triplets = json.load(f)

print(f"Triplets cargados: {len(all_triplets)}")

In [ ]:
# ============================================================================
# 17. FILTRAR TRIPLETS SOLO DE TRAIN
# ============================================================================

train_triplets = [
    t for t in all_triplets
    if t.get("positive_chapter_order") in train_chapter_orders
    and t.get("negative_chapter_order") in train_chapter_orders
]

print(f"Triplets de train tras filtrar por capítulo: {len(train_triplets)}")

if len(train_triplets) == 0:
    raise ValueError("No hay triplets de entrenamiento tras el filtrado.")

print("\nPreview rápido:")

for t in random.sample(train_triplets, min(2, len(train_triplets))):
    print("\nA:", t["anchor"])
    print("P capítulo:", t["positive_chapter_title"])
    print("P:", t["positive"][:180], "...")
    print("N capítulo:", t["negative_chapter_title"])
    print("N:", t["negative"][:180], "...")

In [ ]:
for var_name in ["model", "emb_model", "finetuned_model"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================================
# 18. FINE-TUNING DEL MODELO DE EMBEDDINGS
# ============================================================================

from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

# Liberar memoria de posibles modelos anteriores
for var_name in ["model", "emb_model", "finetuned_model"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Entrenando en: {device}")

train_examples = [
    InputExample(
        texts=[
            t["anchor"],
            t["positive"],
            t["negative"]
        ]
    )
    for t in train_triplets
]

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=FT_BATCH_SIZE
)

finetuned_model = SentenceTransformer(
    EMBEDDING_NAME,
    device=device
)

train_loss = losses.TripletLoss(finetuned_model)

warmup_steps = max(
    1,
    int(len(train_dataloader) * FT_EPOCHS * FT_WARMUP_RATIO)
)

print(f"Warmup steps: {warmup_steps}")
print(f"Batch size:   {FT_BATCH_SIZE}")
print(f"Epochs:       {FT_EPOCHS}")
print(f"LR:           {FT_LEARNING_RATE}")

finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=FT_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": FT_LEARNING_RATE},
    show_progress_bar=True,
    output_path=FINETUNED_MODEL_PATH
)

print(f"\nModelo guardado en: {FINETUNED_MODEL_PATH}")

In [ ]:
# ============================================================================
# 19. EVALUAR MODELO FINE-TUNEADO
# ============================================================================

finetuned_results = evaluate_model(
    FINETUNED_MODEL_PATH,
    "fine_tuned"
)

In [ ]:
# ============================================================================
# 20. COMPARACIÓN FINAL
# ============================================================================

print("\nComparación final:")

ordered_metrics = sorted(
    baseline_results.keys(),
    key=lambda x: (x.split("@")[0], int(x.split("@")[1]))
)

for metric in ordered_metrics:
    base = baseline_results[metric]
    ft = finetuned_results[metric]
    delta = ft - base

    print(
        f"{metric}: "
        f"baseline={base:.4f} | "
        f"fine_tuned={ft:.4f} | "
        f"delta={delta:+.4f}"
    )

Como vemos, el modelo fine-tuneado es peor en todas las métricas que el modelo base.

Esto probablemente se deba a que, o bien lso datos de entrenamiento que tenemos no son especialmente buenos (es decir, los triplets no son lo suficientemente buenos), o son demasiado genéricos.

Por eso mismo, hemos decidido que, en base a los resultados obtenidos, vamos a seguir utilizando el modelo de embeddings base.## Comparación entre modelo base y modelo fine-tuneado

Para evaluar el rendimiento del sistema de recuperación se han utilizado dos métricas principales: **MRR@k** y **Recall@k**, calculadas para distintos valores de `k` (`1`, `3`, `5` y `10`). El objetivo es comparar el comportamiento del modelo de embeddings original frente al modelo ajustado mediante fine-tuning.

---

### Métricas utilizadas

#### MRR@k — Mean Reciprocal Rank at k

El **MRR@k** mide en qué posición aparece el primer resultado relevante dentro de los `k` primeros documentos recuperados.

Su valor es mayor cuando el primer documento relevante aparece en posiciones más altas del ranking. Por ejemplo, si el primer documento relevante aparece en la primera posición, la contribución es `1`; si aparece en la segunda posición, la contribución es `1/2`; si aparece en la tercera, `1/3`, etc.

En este caso, MRR@k permite evaluar si el sistema coloca rápidamente un chunk relevante entre los primeros resultados.

---

#### Recall@k

El **Recall@k** mide qué proporción de los documentos relevantes se recupera dentro de los `k` primeros resultados.

Un Recall@k alto indica que el sistema está encontrando más documentos relevantes dentro del conjunto de resultados devueltos. A diferencia del MRR, no se centra tanto en la posición exacta del primer resultado relevante, sino en cuántos resultados relevantes aparecen dentro del top-k.

---

## Resultados obtenidos

| Métrica | Modelo base | Modelo fine-tuneado | Diferencia |
|---|---:|---:|---:|
| MRR@1 | 0.4541 | 0.1159 | -0.3382 |
| MRR@3 | 0.5628 | 0.1844 | -0.3784 |
| MRR@5 | 0.5763 | 0.2066 | -0.3697 |
| MRR@10 | 0.5840 | 0.2210 | -0.3630 |
| Recall@1 | 0.4541 | 0.1159 | -0.3382 |
| Recall@3 | 0.6957 | 0.2754 | -0.4203 |
| Recall@5 | 0.7536 | 0.3720 | -0.3816 |
| Recall@10 | 0.8116 | 0.4783 | -0.3333 |

---

## Interpretación de los resultados

Los resultados muestran que el modelo base obtiene mejores métricas que el modelo fine-tuneado en todos los casos evaluados.

En **MRR@k**, el modelo base supera claramente al modelo fine-tuneado. Esto indica que el modelo base coloca el primer resultado relevante en posiciones más altas del ranking, mientras que el modelo fine-tuneado tiende a desplazar los resultados relevantes a posiciones inferiores o directamente no los recupera dentro del top-k.

La diferencia también es clara en **Recall@k**. Por ejemplo, en `Recall@10`, el modelo base alcanza un valor de `0.8116`, mientras que el modelo fine-tuneado obtiene `0.4783`. Esto significa que el modelo base recupera una proporción considerablemente mayor de documentos relevantes dentro de los 10 primeros resultados.

La caída más pronunciada aparece en `Recall@3`, con una diferencia de `-0.4203`, lo que indica que el modelo fine-tuneado pierde bastante capacidad para recuperar documentos relevantes en las primeras posiciones del ranking.

---

## Conclusión

El fine-tuning realizado no ha mejorado el rendimiento del modelo de embeddings. Al contrario, ha empeorado tanto la capacidad de ranking como la capacidad de recuperación.

Por tanto, para este sistema RAG, resulta más conveniente mantener el **modelo base** en lugar del modelo fine-tuneado. El modelo base ofrece resultados más sólidos, recupera más documentos relevantes y posiciona mejor los chunks útiles dentro de los primeros resultados.